# 01: Datasets and Preprocessing

Visual exploration of the REIMS datasets using polished dashboard-style charts.

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
import plotly.io as pio
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, precision_recall_curve, average_precision_score
from fishy import TrainingConfig, run_unified_training, display_final_summary, create_data_module
pio.renderers.default = "notebook_connected"


In [ ]:
dm = create_data_module("species"); dm.setup(); df = dm.get_filtered_dataframe()
X, y = dm.get_numpy_data(labels_as_indices=True); names = dm.get_class_names()
feature_cols = [c for c in df.columns if c not in ["Class Name", "m/z"]]
try: mz_axis = np.array([float(c) for c in feature_cols])
except: mz_axis = np.arange(len(feature_cols))
print(f"Loaded {len(df)} samples.")


### Mean Spectral Signatures

In [ ]:
fig = go.Figure()
for i, name in enumerate(names):
    mask = (y == i)
    if mask.any():
        m, s = X[mask].mean(axis=0), X[mask].std(axis=0)
        fig.add_trace(go.Scatter(x=mz_axis, y=m, name=name))
        fig.add_trace(go.Scatter(x=np.concatenate([mz_axis, mz_axis[::-1]]), y=np.concatenate([m+s, (m-s)[::-1]]), fill="toself", fillcolor="rgba(0,100,80,0.1)", line=dict(color="rgba(255,255,255,0)"), showlegend=False))
fig.update_layout(template="plotly_white", xaxis_title="m/z", yaxis_title="Intensity").show()


### Dimensionality Reduction

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2); X_p = pca.fit_transform(X)
px.scatter(x=X_p[:,0], y=X_p[:,1], color=[names[i] for i in y], title="PCA Projection", template="plotly_white").show()
